## Managing Tool Permissions

# Unit 10: Granular Authorization Modeling — Designing Advanced Tool Guardrails

## Introduction

Welcome back to **Advanced Claude Code Features**! This is our second unit, and we are making excellent progress. In the previous lesson, we explored legacy custom slash-commands, analyzing both their structure and historical architectural limitations. Now, we turn our attention to a fundamental aspect of working safely with terminal-based workflows: **managing tool permissions**.

In this lesson, we will learn how to tightly govern which local tools Claude can access and under what structural conditions. While Claude's agentic engineering runtime is highly capable, operations like destructive file mutations or raw shell executions carry inherent system risks. We will discover how to inspect live permissions via interactive prompts, configure automatic approvals for low-risk utilities, engineer pattern-based rules, and decode the evaluation priority hierarchy that governs these constraints.

By the end of this module, you will know how to design a balanced permission topology—ensuring Claude can execute routine terminal tasks with zero friction while insulating your production source code from unintended modifications.

---

## Why Tool Permissions Matter

Before diving into configuration arrays, let's look at why automated permissions management is essential. Claude Code provides an array of underlying system tools that interface directly with your host environment: reading file lines, editing blocks, searching buffers, and executing terminal subprocesses. While these capabilities empower seamless debugging, they also introduce functional risk.

Consider the structural impact if an agent runs completely unconstrained: Claude might accidentally wipe out an uncommitted file, run an unescaped destructive loop, or make external network packets that expose internal API secrets. Automated tasks carry inherent risk, which is why structured tool permissions serve as critical architectural guardrails.

The objective is to establish clear boundaries, not to introduce unnecessary roadblocks. Safe diagnostic operations (like reading or searching code patterns) can fire automatically, while high-risk actions (like deleting paths or pushing to remote branches) should be constrained by strict manual approvals. This balance optimizes developer velocity while maintaining safety.

---

## Categorizing Structural Tool Risks

Not all system tools present the same risk profile. Claude Code groups its internal executable capabilities into distinct buckets based on their potential impact on your workspace:

### 🟢 Low-Risk Tools (Read-Only Operations)

These tools perform actions that only retrieve or search text data without modifying system state. They are safe to auto-approve because they cannot alter your code or environment:

* **`Read`**: Allows the agent to view raw file lines across the repository.
* **`WebSearch`**: Performs external web lookups to fetch modern documentation frames.
* **`Grep`**: Executes localized text searches across your project files.
* **`Glob`**: Maps workspace contents matching precise path patterns.

### 🔴 Medium to High-Risk Tools (State-Mutating Operations)

These tools alter the state of files, write new files, or drop into system shell threads. They require careful observation:

* **`Edit`**: Alters file blocks by generating and applying localized patches.
* **`Write`**: Instantiates brand-new files or overwrites existing path definitions.
* **`Bash`**: Spawns an interactive shell process with full host permissions.

A single unverified `Edit` can break critical production logic, and an unescaped `Bash` execution can alter system packages or wipe data. For these reasons, we want manual approval for these capabilities, at least initially.

---

## Navigating the Interactive Permissions UI

Claude Code provides an interactive panel to inspect and customize your running permission matrix. You can access this layout at any time by entering the `/permissions` slash-command:

```text
> /permissions
─────────────────────────────────────────────────────────────────────────
 Permissions:  Allow   Ask   Deny   Workspace  (←/→ or tab to cycle)

 Claude Code will always ask for confirmation before using these tools.
 ╭────────────────────────────────────────────────────────────────────╮
 │ ⌕ Search…                                                          │
 ╰────────────────────────────────────────────────────────────────────╯

   1. Add a new rule…

 Press ↑↓ to navigate · Enter to select · Type to search · Esc to cancel

```

This interactive interface groups your permissions into four view tabs:

* **`Allow`**: Explicit lists of tools authorized to run automatically without confirmation boxes.
* **`Ask`**: Tools requiring manual developer confirmation before execution.
* **`Deny`**: Tool vectors blocked from executing under any condition.
* **`Workspace`**: Project-specific rules constrained purely to the current repository directory.

By default, Claude Code automatically processes permissions based on risk level. Read-only tools like `Read` or `Grep` bypass confirmation prompts to keep your development workflow smooth. In contrast, destructive or state-mutating actions like `Write`, `Edit`, or raw shell execution require manual confirmation. You can use the **`Add a new rule...`** action to fine-tune these properties to your specific workflow.

---

## Configuring Auto-Approval for Low-Risk Tools

To optimize velocity during deep code exploration, you can declare safe tools to execute automatically. This is managed within the `permissions` configuration block inside your project's `.claude/settings.json` file:

```json
{
  "permissions": {
    "allow": ["Read", "WebSearch", "Grep", "Glob"]
  }
}

```

By adding these tools to the `allow` array, operations like reading file trees or pulling web documentation run instantly in the background without populating your screen with confirmation dialogue boxes.

---

## Designing Target Confirmation Rules via Glob Matching

For tools that present higher operational risk, you can leverage the `ask` array to mandate a user confirmation prompt. This block supports **glob-style syntax** matching for highly precise, structural enforcement over paths and patterns:

```json
{
  "permissions": {
    "allow": ["Read", "WebSearch", "Grep", "Glob"],
    "ask": [
      "Edit(**/*.ts)",
      "Edit(**/*.tsx)",
      "Bash(rm*)"
    ]
  }
}

```

### Deconstructing the Glob Targets:

* **`Edit(/*.ts)` & `Edit(/*.tsx)**`: Forces a confirmation dialog box if Claude attempts to apply code edits to any TypeScript or React files across any nested repository folders.
* **`Bash(rm*)`**: Halts any shell command that begins with the removal command string, requiring explicit confirmation.

> 📝 **Glob Syntax Rules:** The `*` wildcard matches variable characters within a single path segment, whereas the double wildcard `**` recursively evaluates any depth of nested folders. This allows you to protect critical application branches while keeping routine adjustments frictionless.

---

## Configuration Defenses: The Limitations of `settings.json`

While the file system configuration specification supports a `deny` block designed to flag blocked operations:

```json
{
  "permissions": {
    "deny": [
      "Bash(rm*)",
      "Bash(sudo)",
      "Edit(**/.env)",
      "Edit(**/.env.*)"
    ]
  }
}

```

> ⚠️ **Critical Security Warning:** In early framework versions, file-level `deny` array blocks are primarily advisory and **are not reliably enforced at the `settings.json` layer**. The model may still execute operations declared within a file-based deny block. For reliable, immutable guardrails, **you must use CLI flags at session launch**.

---

## Strong Enforcement: Utilizing Launch Flags

To enforce strict security constraints on a session, you must pass permission configuration flags directly into the terminal command line when launching Claude Code:

```bash
claude --allowedTools "Read,Write" --disallowedTools "Bash(rm*)"

```

When initialized with these parameters, the session activates with hard system limits:

```text
╭─── Claude Code v2.1.17 ──────────────────────────────────╮
│            Welcome back!                                 │
│   Sonnet 4.5 · API Usage Billing                         │
│        /usercode/FILESYSTEM                              │
╰──────────────────────────────────────────────────────────╯

```

Unlike declarative json settings parameters, environment constraints applied via CLI flags **cannot be bypassed**. If the model attempts to invoke a forbidden utility string, the local tool bus rejects the transaction instantly:

```text
> Run: rm test.txt
● Bash(rm test.txt)
  ⎿ Error: Permission to use Bash with command rm test.txt has been denied.

● The rm test.txt command was blocked by your permissions configuration.

```

These launch flags provide secure permission control for highly specific operational goals. For example, during a routine documentation pass, you might lock down the entire agent environment to only permit file modifications while completely disabling shell subprocess access.

---

## Evaluating the Permission Priority Chain

When multiple flags, pattern rules, and local settings are configured simultaneously, Claude Code resolves the active authorization state using a strict, safety-first priority model:

```text
 Evaluation Precedence Matrix
 ─────────────────────────────────────────────────────────────
  1. --disallowedTools   ──► Blocks instantly, overriding all.
  2. permissions.ask     ──► Prompts for explicit user approval.
  3. --allowedTools      ──► Authorizes background execution.

```

The specific `--disallowedTools` command-line pattern will always take precedence over broader allowance configurations. If you boot up Claude Code using this composite parameters signature:

```bash
claude --allowedTools "Bash" --disallowedTools "Bash(rm*)"

```

The tool broker evaluates the incoming rules from most restrictive to least restrictive. Routine diagnostics like running a directory list (`Bash(ls -la)`) will fire instantly in the background without interruptions, but any destructive invocation like `Bash(rm file.txt)` is automatically blocked. Safety conditions cannot be accidentally bypassed by broad global permissions.

---

## Auto-Approved Invocations in Action

Once your `allow` arrays are configured, safe tasks execute seamlessly in the background. For example, when asking Claude Code to pull package changes, it fires its internet access utilities immediately without showing an alert box:

```text
> Search for the latest pandas version
● Web Search("latest pandas version 2026")
  ⎿ Did 1 search in 15s

● The latest pandas version is pandas 3.0.0, released on January 21, 2026.

```

The tool bypasses confirmation screens because `WebSearch` is explicitly authorized in the permission profile. This creates a fast, automated workflow for safe tasks while protecting sensitive actions with the `ask` array and CLI launch flags.

---

## Summary of Permission Configurations

| Control Strategy Layer | Implementation Vector | Real-World Security Use Case | Core Enforcement Level |
| --- | --- | --- | --- |
| **`allow` Array** | `.claude/settings.json` | Automates routine read tasks like `Read` and `Grep`. | Advisory / Fluid Workflow |
| **`ask` Glob Matching** | `.claude/settings.json` | Pauses for review when editing core scripts (e.g., `Edit(**/*.ts)`). | Advisory / Task Warning |
| **`--allowedTools`** | CLI Terminal Flag | Authorizes specific tools for an isolated development session. | **Strictly Enforced** |
| **`--disallowedTools`** | CLI Terminal Flag | Immutably blocks destructive actions like `Bash(rm*)` or `Bash(sudo)`. | **Strictly Enforced (Top Priority)** |

---

## Conclusion

Managing tool permissions means establishing the right balance for your project—automating safe tasks to speed up development while setting strict confirmation boundaries for actions that modify the system. The local `settings.json` file is perfect for configuring workflow preferences and team patterns, while terminal-level CLI launch flags give you guaranteed security enforcement.

Now, let's put these configuration paradigms to work! The upcoming lab challenges will guide you through setting up secure permission configurations for various production scenarios. Turn the page and let's get started.

## Configuring Auto Approval for Safe Tools

Let's start by exploring Claude's permission system from the inside out. Start a new conversation with Claude in your terminal, type the /permissions command and press enter. Take a moment to navigate through the tabs (Allow, Ask, Deny, Workspace) using arrow keys or tab. Notice that they're all empty in the default configuration — this doesn't mean Claude asks for approval before using any tool, but rather that no custom permission rules have been configured yet. In practice, Claude intelligently auto-approves safe, read-only operations like Read, Search, and certain non-destructive Bash commands, while prompting for operations that could modify your workspace.

Now it's time to make your workflow even more efficient by explicitly configuring automatic approvals for additional safe tools.

Ask Claude to create or edit the .claude/settings.json file to enable auto-approval for safe, read-only tools. The configuration should use the allow field inside the permissions object. Specifically, request that Claude add these four low-risk tools to the allow array:

    Read
    WebSearch
    Grep
    Glob

Once Claude has made this configuration change, exit the current Claude session and start Claude again. This restart is necessary for the permission changes to take effect — the /permissions interface only reflects settings that were loaded at startup.

After restarting Claude, run /permissions again and navigate to the Allow tab — you should now see your four configured tools listed there!

Finally, test it by asking Claude to perform a safe operation — try searching the web for something or reading a file from your project. You should notice that these operations now execute immediately without any approval prompts appearing.

This is your first step toward balancing security with convenience, letting harmless read-only operations flow smoothly while keeping the guardrails in place for riskier actions.

### Step 1: Open the Interactive Permissions Dashboard

```text
/permissions

```

*(Use your arrow keys or **Tab** to view the tabs, then press **Esc** to close the overlay).*

---

### Step 2: Configure the Auto-Approval Tool Settings

```text
Create a new file at .claude/settings.json with an allow array inside the permissions object containing: "Read", "WebSearch", "Grep", and "Glob".

```

*(Select **`1. Yes`** to approve the file modification).*

---

### Step 3: Reload the Workspace Session

```text
/exit

```

```bash
claude

```

---

### Step 4: Verify the Loaded Configuration

```text
/permissions

```

*(Navigate to the **Allow** tab to verify your four tools are listed, then press **Esc** to close).*

---

### Step 5: Test the Frictionless Tool Workflow

```text
Search the web for the latest Python release features.

```

---

### Step 6: Finalize and Submit

```text
/submit

```

## Adding Pattern Based Confirmation Rules

Now that you've set up auto-approval for safe tools, let's add more precision by creating rules that target specific operations.

Ask Claude to enhance your .claude/settings.json file by adding a permissions.ask configuration with pattern-based rules. Request patterns that require approval for:

    Editing TypeScript files (**/*.ts and **/*.tsx)
    Running file deletion commands (rm*)

Important: After Claude configures these rules, you'll need to exit the current Claude session and start it again for the changes to appear in the /permissions tab.

After restarting Claude with the new configuration, test the rules using the example.ts file in your workspace:

    Test editing — Ask Claude to modify example.ts (for example, add a new function or update the interface). Verify that an approval prompt appears.

    Test reading — Ask Claude to read or display the contents of example.ts. Verify that this operation executes automatically without prompting (since Read is in your allowed operations).

This demonstrates granular control — automatically approving routine read operations while adding checkpoints for operations that could modify your codebase. The permissions.ask configuration creates explicit safety boundaries that persist even if you later broaden your permissions.

To apply these precise pattern-based configuration rules and verify how they manage your tools differently, follow this exact step-by-step input loop:

### Step 1: Update the Configuration File with Confirmation Rules

Copy and paste this explicit update request into your `>` prompt and press **Enter**:

```text
Update .claude/settings.json to add an "ask" array inside the permissions object containing these three pattern-matching rules: "Edit(**/*.ts)", "Edit(**/*.tsx)", and "Bash(rm*)".

```

* **Authorize:** When the terminal displays the file change diff window highlighting the new `"ask"` lines, select **`1. Yes`** to let the `Edit` tool patch your settings file.

---

### Step 2: Reload Your Session to Activate the Rules

Type the exit shortcut to leave your current session:

```text
/exit

```

Relaunch Claude Code in your terminal to initialize a fresh window with the newly updated permission files loaded:

```bash
claude

```

---

### Step 3: Test Modifying (Enforcing the `ask` Pattern Guardrail)

Once the interactive prompt loads, issue an update request targeting your TypeScript file:

```text
Add a descriptive comment block to the top of example.ts.

```

* **Verify:** Notice how Claude switches to its `Edit` tool but **explicitly pauses to display a permission confirmation block**. This happens because the file path matches your custom `Edit(/*.ts)` rule perfectly. Select **`1. Yes`** to allow it.

---

### Step 4: Test Reading (Enforcing the frictionless `allow` Filter)

Now, test a read operation on that exact same file to verify your low-risk rules remain active:

```text
Display the contents of example.ts.

```

* **Verify:** Notice the difference! Because `Read` is declared in your `allow` array, Claude reads and streams the file contents onto your screen **instantly with no confirmation prompt**.

---

### Step 5: Finalize and Submit

With both verification test runs successfully logged in your terminal session history, complete the lab module by typing:

```text
/submit

```

By working through this challenge, you have mastered pattern-based permission modeling, keeping routine reading completely automated while guarding code mutations behind strict confirmation checkpoints! 🚀

## Enforcing Permissions with CLI Flags

So far, you've been setting permissions through the .claude/settings.json file, which works well for persistent configurations. But what if you need stronger, session-specific controls that can't be changed mid-conversation? That's where CLI flags come in.

Start Claude from your terminal, but this time add these flags to your launch command:

    --allowedTools "Read,Write" to auto-approve file reading and writing
    --disallowedTools "Bash(rm*)" to block any bash commands that start with rm

Once Claude starts up in this new session, test your permissions by first asking Claude to perform allowed operations — try reading a file or writing to one. These should execute immediately without any prompts.

Next, ask Claude to remove a file using a command such as rm example.ts. You should see an immediate denial with an error message — no approval prompt will even appear because the operation is completely blocked at the session level.

This exercise shows you how CLI flags create an unbreakable permission layer that settings files can't override, giving you rock-solid control when you need it.

To enforce strict session-level guardrails that cannot be altered or bypassed mid-conversation, you must pass permission boundaries as flags directly when launching your session.

Because CLI launch flags must be executed from your host system shell, follow this step-by-step sequence to leave your active interactive window, launch with constraints, and test the enforcement:

### Step 1: Drop Back to Your Terminal Shell

Exit your current Claude session to return to your standard command prompt (where you see `/usercode/FILESYSTEM$`):

```text
/exit

```

---

### Step 2: Relaunch Claude Code with Immutable Security Flags

Launch a new session by copying and pasting this flag-configured command directly into your terminal shell and hitting **Enter**:

```bash
claude --allowedTools "Read,Write" --disallowedTools "Bash(rm*)"

```

---

### Step 3: Test Allowed Operations (Frictionless Execution)

Once the interactive session initializes, test your auto-approved tools by asking Claude to modify a text file:

```text
Write a file named flag_test.txt containing the text "Testing launch flags."

```

* **Verify:** Because `Write` was explicitly declared in your `--allowedTools` parameter flag, Claude instantly initializes the file in the background without prompting you with a confirmation window.

---

### Step 4: Test Disallowed Operations (Strict Enforcement Block)

Now, attempt to execute a forbidden deletion string to see the priority engine intercept the tool broker:

```text
Run this command: rm flag_test.txt

```

* **Verify:** Notice the immediate system rejection! Instead of bringing up an interactive prompt box to ask for your permission, the shell broker blocks the operation entirely at the session level and prints a strict error:
`⎿ Error: Permission to use Bash with command rm flag_test.txt has been denied.`

---

### Step 5: Finalize and Submit

With the unbreakable session flag trace successfully written to your lab environment log, complete this security module by typing:

```text
/submit

```

By completing this exercise, you have mastered launch-level permission molding—ensuring that high-risk terminal commands are locked down completely from the moment the agent initializes! 🚀

## Understanding Permission Priority with Overlapping Rules